<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/TRansformersFAkeReal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
from google.colab import userdata
import cv2
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image
from google.colab import files
from sklearn.metrics import classification_report
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

print("Kaggle API activated via Secrets!")

Kaggle API activated via Secrets!


In [ ]:

# Paste the copied command here, starting with an exclamation mark
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:23<00:00, 168MB/s]



In [ ]:
!unzip -q 140k-real-and-fake-faces.zip -d ./dataset_folder

In [ ]:
train_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/train'
test_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/test'
validation_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/valid'

In [ ]:
train_gen = ImageDataGenerator(rescale= 1./255)
test_gen = ImageDataGenerator(rescale= 1./255)
validation_gen = ImageDataGenerator(rescale= 1./255)

In [ ]:
train_data = train_gen.flow_from_directory(train_dir,target_size=(128,128),batch_size=32,class_mode='binary')
test_data = test_gen.flow_from_directory(test_dir,target_size=(128,128),batch_size=32,class_mode='binary')
validation_data = validation_gen.flow_from_directory(validation_dir,target_size=(128,128),batch_size=32,class_mode='binary')

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [ ]:
image_size = 128
patch_size = 8
num_patches = (image_size // patch_size) ** 2  # divide imagesize with patchsize and square of that value
embed_dim = 256
num_heads = 8
ff_dims = 150

In [ ]:
def vit_input(inputs):
  #Brick 1 for token
  #chop the images into 16x16
  x = layers.Conv2D(filters=embed_dim,kernel_size=patch_size,strides=patch_size,padding='valid')(inputs)

  #Here after above code we have (batch size,8,8,64). and we have to convert it back to (batch size, 64,64)
  x = layers.Reshape((num_patches,embed_dim))(x) # this is basically the token embedding

  # Brick 1 for position embedding
  position_range = tf.range(start=0,limit=num_patches,delta=1)
  position_embed = layers.Embedding(input_dim=num_patches,output_dim=embed_dim)(position_range)

  return x + position_embed

In [ ]:
def Transformer_Block(inputs):
  #Brick2: Multihead QKV
  attention_output = layers.MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim)
  attention_output = attention_output(query=inputs,key=inputs,value=inputs)
  attention_output = layers.Dropout(0.1)(attention_output)

  #Brick4: Glue
  x = layers.LayerNormalization(epsilon=1e-6)(inputs + attention_output)

  #Brick3: MLP
  x2 = layers.Dense(ff_dims,activation='relu')(x)
  x2 = layers.Dense(embed_dim)(x2)
  x2 = layers.Dropout(0.1)(x2)

  return layers.LayerNormalization(epsilon=1e-6)(x + x2)

In [ ]:

# 1. Define the "Safety Net"
early_stop = EarlyStopping(
    monitor='val_loss',   # Watch the validation loss
    patience=4,           # How many epochs to wait for improvement before quitting
    restore_best_weights=True # Keep the version of the model that performed best
)

In [ ]:

# 2. The Sharpener (Shrinks LR if stuck)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,             # Cut LR by 80% (make it much smaller)
    patience=2,             # Wait 2 epochs before shrinking
    min_lr=1e-6             # Don't let it get smaller than this
)

In [ ]:
my_callbacks = [early_stop, reduce_lr]

In [ ]:
inputs = layers.Input(shape=(image_size,image_size,3))

#Entrance to convert images to vectors
x1 = vit_input(inputs)

block_x1 = Transformer_Block(x1)
block_x1 = Transformer_Block(x1)
block_x1 = Transformer_Block(x1)
block_x1 = Transformer_Block(x1)
block_x1 = Transformer_Block(x1)
block_x1 = Transformer_Block(x1)

x = layers.GlobalAveragePooling1D()(block_x1)
x = layers.Dense(128,activation='relu')(x)
x = layers.Dropout(0.3)(x)

output = layers.Dense(1,activation='sigmoid')(x)

model = tf.keras.Model(inputs=inputs,outputs=output)

model.compile(optimizer= 'adam',loss= 'binary_crossentropy',metrics=['accuracy'])

model.fit(train_data,validation_data=validation_data,epochs=15,callbacks=my_callbacks)

Epoch 1/15
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 233s 72ms/step - accuracy: 0.5247 - loss: 0.6906 - val_accuracy: 0.5321 - val_loss: 0.6915 - learning_rate: 0.0010
Epoch 2/15
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 221s 71ms/step - accuracy: 0.5428 - loss: 0.6857 - val_accuracy: 0.5889 - val_loss: 0.6704 - learning_rate: 0.0010
Epoch 3/15
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 219s 70ms/step - accuracy: 0.5827 - loss: 0.6737 - val_accuracy: 0.6087 - val_loss: 0.6578 - learning_rate: 0.0010
Epoch 4/15
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 219s 70ms/step - accuracy: 0.5815 - loss: 0.6729 - val_accuracy: 0.5951 - val_loss: 0.6646 - learning_rate: 0.0010
Epoch 5/15
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 219s 70ms/step - accuracy: 0.6054 - loss: 0.6621 - val_accuracy: 0.5972 - val_loss: 0.6651 - learning_rate: 0.0010
Epoch 6/15
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 219s 70ms/step - accuracy: 0.6125 - loss: 0.6553 - val_accuracy: 0.6179 - val_loss: 0.6503 - learning_rate: 2.0000e-04
Epoch 7/15
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 216s 69ms/st